## The State of Tax Justice: Estimate misalignment - Mario's estimates

- Author: Mario Cuenda García, based on Alison Schultz, based on Javier Garcia Bernado's work
- Created: 4 November 2024
- Last updated: 22 September 2024

**Description**
- This notebook is the third out of three notebooks to estimate the tax losses caused by profit shifting by multinational enterprises (MNEs). The analysis used the misalignment method based on the country-by-country reports (CbCR) published by the OECD.
    - Details on the misalignment method and its background can be found here: https://www.sciencedirect.com/science/article/pii/S0305750X23003455. 
    - The working paper version is here: https://www.econstor.eu/bitstream/10419/286362/1/wp-2023-33.pdf 

- This notebook estimates profit misalignment based on different formulas. It uses the dataset **"data/final/cbcr_main.csv"** (for the estimation with imputed values) or the dataset **"data/final/cbcr_main_noimputation_allsubgroupsonly.csv"** (for the estimation without imputed values). 

**Outline**
1. Define misalignment
2. Calculate misalignment for sample with full information.
3. Calculate misalignment for samples with imputed data and aggregate results. 

**To dos before running this notebook**
- Run the notebooks 1_clean and 2_impute_missings. Note the requirements and instructions given in these notebooks.

**To dos in this notebook**

Change the formula to any required formula in each section and adapt the output name of the csvs. The formulas I have used are like follows, where sales refer to unrelated party revenues and assets to tangible assets excluding cash.

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]

- Adjust the input path in 5.1 to the bootstrapped sample you use

## 0. Load packages

In [1]:
# Packages
import pandas as pd
import numpy as np
import tjn_tools
from config import *

# Show columns and select data format
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.0f}'.format

[TJN TOOLS: Data processing] Module loaded.
[TJN TOOLS: Other functions] Module loaded.
[TJN TOOLS: Paths] Module loaded. Sharepoint FOUND at /Users/mariocuendagarcia/Library/CloudStorage/OneDrive-SharedLibraries-TaxJusticeNetworkLtd


## Step x. Generate the dataset with Unique ISO parents

In [2]:
# Open the original dataset
iso_parents = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Keep just the following columns: iso_parent and year
iso_parents = iso_parents[['iso_parent', 'year']]

# Keep every unique combination of iso_parent and year
iso_parents = iso_parents.drop_duplicates(subset=['iso_parent', 'year'])

# Sort by year, then iso_parent
iso_parents = iso_parents.sort_values(by=['year', 'iso_parent'])

# Filter by year
iso_parents_2016= iso_parents[iso_parents['year'] == 2016]
iso_parents_2017= iso_parents[iso_parents['year'] == 2017]
iso_parents_2018= iso_parents[iso_parents['year'] == 2018]
iso_parents_2019= iso_parents[iso_parents['year'] == 2019]
iso_parents_2020= iso_parents[iso_parents['year'] == 2020]
iso_parents_2021= iso_parents[iso_parents['year'] == 2021]

# OPTIONAL: Print the count of how many unique iso_partner values there are
print(iso_parents_2016['iso_parent'].nunique())
print(iso_parents_2017['iso_parent'].nunique())
print(iso_parents_2018['iso_parent'].nunique())
print(iso_parents_2019['iso_parent'].nunique())
print(iso_parents_2020['iso_parent'].nunique())
print(iso_parents_2021['iso_parent'].nunique())

iso_parents_2016

26
38
46
50
52
52


,iso_parent,year
264,AUS,2016
760,AUT,2016
833,BEL,2016
1063,BMU,2016
1644,BRA,2016
1893,CAN,2016
2722,CHL,2016
2816,CHN,2016
4968,DNK,2016
6205,FIN,2016


## Step x. Generate the dataset with unique iso_partners

In [3]:
## Download the relevant dataset
iso_partners = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
iso_partners = iso_partners[~iso_partners['iso_partner'].isin(non_countries)]

# Keep just the following columns: iso_partner and year
iso_partners = iso_partners[['iso_partner', 'year']]

# Sort by year, then iso_partner
iso_partners = iso_partners.sort_values(by=['year', 'iso_partner'])

# Keep every unique combination of iso_partner and year
iso_partners = iso_partners.drop_duplicates(subset=['iso_partner', 'year'])

# Filter by year
iso_partners_2016= iso_partners[iso_partners['year'] == 2016]
iso_partners_2017= iso_partners[iso_partners['year'] == 2017]
iso_partners_2018= iso_partners[iso_partners['year'] == 2018]
iso_partners_2019= iso_partners[iso_partners['year'] == 2019]
iso_partners_2020= iso_partners[iso_partners['year'] == 2020]
iso_partners_2021= iso_partners[iso_partners['year'] == 2021]


# Optional: Print the count of how many unique iso_partner values there are 
print(iso_partners_2016['iso_partner'].nunique())
print(iso_partners_2017['iso_partner'].nunique())
print(iso_partners_2018['iso_partner'].nunique())
print(iso_partners_2019['iso_partner'].nunique())
print(iso_partners_2020['iso_partner'].nunique())
print(iso_partners_2021['iso_partner'].nunique())

183
215
213
210
212
211


## Step x. Generate the template dataset

In [4]:
# Perform a cross join to merge all values of iso_partners_ with each value of iso_parents
iso_combinations_2016 = iso_parents_2016.assign(key=1).merge(iso_partners_2016.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2017 = iso_parents_2017.assign(key=1).merge(iso_partners_2017.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2018 = iso_parents_2018.assign(key=1).merge(iso_partners_2018.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2019 = iso_parents_2019.assign(key=1).merge(iso_partners_2019.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2020 = iso_parents_2020.assign(key=1).merge(iso_partners_2020.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2021 = iso_parents_2021.assign(key=1).merge(iso_partners_2021.assign(key=1), on='key').drop('key', axis=1)

# Concatenate all the years
template_dataset = pd.concat([iso_combinations_2016, iso_combinations_2017, iso_combinations_2018, iso_combinations_2019, iso_combinations_2020, iso_combinations_2021])

# Drop year_y
template_dataset = template_dataset.drop(columns=['year_y'])
# Rename year_x to year
template_dataset = template_dataset.rename(columns={'year_x': 'year'})
# Order columns by iso_parent then iso_partner then year
template_dataset = template_dataset[['iso_parent', 'iso_partner', 'year']]
# Generate new column called cbcr_estimates
template_dataset['cbcr_estimates'] = np.nan

template_dataset

,iso_parent,iso_partner,year,cbcr_estimates
0,AUS,ABW,2016,NaN
1,AUS,AFG,2016,NaN
2,AUS,AGO,2016,NaN
3,AUS,ALB,2016,NaN
4,AUS,AND,2016,NaN
...,...,...,...,...
10967,ZAF,XKV,2021,NaN
10968,ZAF,YEM,2021,NaN
10969,ZAF,ZAF,2021,NaN
10970,ZAF,ZMB,2021,NaN


## 1. Define misalignment

In [5]:
def calculate_misalignment(cbcr_data,
                           formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",
                                         'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip'],
                           weights=[.5, 0, 0, .5, 0, 0, 0, 0],
                           profit_var='profit_loss_before_income_tax_corrected',
                           etr_max=0.15): 

    # Create variable with positive profits only for calculating shares
    cbcr_data['profit_var_pos'] = cbcr_data[profit_var]
    cbcr_data.loc[cbcr_data[profit_var] < 0, 'profit_var_pos'] = 0
    cbcr_data['share_profit'] = cbcr_data['profit_var_pos'] / cbcr_data.groupby('iso_parent')['profit_var_pos'].transform('sum')

    # Calculate weighted shares of economic activity
    actual_weights = []
    actual_variables = []
    for i, var in enumerate(formula_vars):
        if var is not None and weights[i] > 0:
            actual_variables.append(f"share_{var}")
            actual_weights.append(weights[i])
            cbcr_data.loc[cbcr_data[var] < 0, var] = 0  # Set economic activity measure to zero if negative
            cbcr_data[f"share_{var}"] = cbcr_data[var] / cbcr_data.groupby('iso_parent')[var].transform('sum')

    # Calculate the share of economic activity
    cbcr_data["share_economy_partner_of_parent"] = (cbcr_data.loc[:, actual_variables] * actual_weights).sum(1, min_count=len(actual_weights))
    # Set economic activity to 1% for those jurisdictions without economic activity but with reported profits
    cbcr_data.loc[(cbcr_data["share_economy_partner_of_parent"] == 0) & (cbcr_data[profit_var] > 0), "share_economy_partner_of_parent"] = 0.01

    # Normalize the economic activity shares to sum to 1
    cbcr_data["share_economy_partner_of_parent"] = cbcr_data["share_economy_partner_of_parent"] / cbcr_data.groupby('iso_parent')["share_economy_partner_of_parent"].transform('sum')

    # Calculate theoretical profit and misaligned profit
    cbcr_data["theoretical_profit"] = cbcr_data["share_economy_partner_of_parent"] * cbcr_data.groupby('iso_parent')[profit_var].transform('sum')
    cbcr_data["misaligned_profit"] = cbcr_data[profit_var] - cbcr_data["theoretical_profit"]

    # Set positive misaligned profits to 0 if ETR exceeds the threshold (etr_max)
    cbcr_data.loc[((cbcr_data["misaligned_profit"] > 0) & (cbcr_data["etr_average_corrected"] > etr_max)), "misaligned_profit"] = 0

    # Adjust misalignment per 'iso_parent'
    def adjust_misalignment(group):
        total_negative_misalignment = group.loc[group["misaligned_profit"] < 0, "misaligned_profit"].sum()
        total_positive_misalignment = group.loc[group["misaligned_profit"] > 0, "misaligned_profit"].sum()
        
        # Adjust negative misalignments to balance positive misalignments within each 'iso_parent'
        if total_negative_misalignment != 0:
            factor = - total_positive_misalignment / total_negative_misalignment
            group.loc[group["misaligned_profit"] < 0, "misaligned_profit"] *= factor
        
        return group

    # Apply the adjustment by grouping by 'iso_parent'
    cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)

    return cbcr_data

## 2. Calculate misalignment for sample with full information

### 2.1 Import data
- Import data without imputed values. This is only the data from the sample of reporting countries that actually is reported on a country basis, i.e. excluding aggregated country groups and data from reporting countries that do not report on a country by country basis, but just by continents.

In [6]:
cbcr_sample = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
cbcr_sample = cbcr_sample[~cbcr_sample['iso_partner'].isin(non_countries)]

### 2.2 Exclude countries that do not report truly country-by-country


- In the 2024 data, the following reporting countries do not report country-by-country. We exclude those from the "clean" analysis where we only use values that are actually in the data.
    - Austria: Only continents in all years
    - Czechia: Only Czechia versus rest of the world from 2019 to 2021
    - Finland: Only Finland and rest of the world between 2016 and 2018 and Finland and continents between 2019 and 2021
    - Greece: Only Greece and continents between 2017 and 2019
    - Hungary: Only Hungary versus rest of the world between 2018 and 2021
    - Isle of Man: Only continents between 2017 and 2020
    - Ireland: Only Ireland versus rest of the world in all years
    - Korea: Only Korea and rest of the world betweem 2016 and 2018 and Korea and continents between 2019 and 2021
    - Macau: Only Macau versus rest of the world between 2019 and 2021
    - Mauritius: Only Mauritius and continents between 2019 and 2021
    - Morocco: Only Morocco and continents in 2021
    - Netherlands: Only Netherlands versus rest of the world between 2016 and 2017
    - Norway: Only Norway and continents 2016 and 2017
    - New Zealand: Only New Zealand versus rest of the world between 2018 and 2021
    - Poland: Only Poland and continents 2019 to 2021
    - Sweden: Only Sweden and continents in all years
    - United Kingdom: Only UK and continents between 2017 and 2021

In [7]:
# Define the conditions for exclusion
exclusion_conditions = [
    ('AUT', 2016, 2021),                # Austria: all years
    ('CZE', 2019, 2021),                # Czechia: from 2019 to 2021
    ('FIN', 2016, 2021),                # Finland: all years
    ('GRC', 2017, 2019),                # Greece: between 2017 and 2019
    ('HUN', 2018, 2021),                # Hungary: between 2018 and 2021
    ('IMN', 2017, 2020),                # Isle of Man: between 2017 and 2020
    ('IRL', 2016, 2021),                # Ireland: all years
    ('KOR', 2016, 2021),                # Korea: all years
    ('MAC', 2019, 2021),                # Macau: between 2019 and 2021
    ('MUS', 2019, 2021),                # Mauritius: between 2019 and 2021
    ('MAR', 2021, 2021),                # Morocco: 2021
    ('NLD', 2016, 2017),                # Netherlands: between 2016 and 2017
    ('NOR', 2016, 2017),                # Norway: 2016 and 2017
    ('NZL', 2018, 2021),                # New Zealand: between 2018 and 2021
    ('POL', 2019, 2021),                # Poland: 2019 to 2021
    ('SWE', 2016, 2021),                # Sweden: all years
    ('GBR', 2017, 2021)                 # United Kingdom: between 2017 and 2021
]

# Iterate through the exclusion conditions
for iso_parent, start_year, end_year in exclusion_conditions:
    cbcr_sample = cbcr_sample[~((cbcr_sample['iso_parent'] == iso_parent) & 
                                     (cbcr_sample['year'].between(start_year, end_year)))]

### 2.3 Calculate misalignment for sample countries with full information

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]


In [8]:
misalignment_2016 = cbcr_sample[cbcr_sample['year'] == 2016].copy()
misalignment_2016 = calculate_misalignment(misalignment_2016, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Keep only the first occurrence of these unique variables for each 'iso_partner'
unique_columns = misalignment_2016.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                               'etr_average_corrected', 'cit',
                                                                               'tax_revenue_current_usd', 
                                                                               'gvt_health_expenditure', 'region_tjn', 
                                                                               'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

# Keep iso_parent, iso_partner, year, misaligned_profit, theoretical_profit, profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
misalignment_2016 = misalignment_2016[['iso_parent', 'iso_partner', 'year', 'misaligned_profit', 'theoretical_profit', 'profit_loss_before_income_tax_corrected', 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']]

# Show if iso_partner = USA
misalignment_2016[misalignment_2016['iso_partner'] == 'USA']

/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_61006/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip
53,AUS,USA,2016,"-5,424,987,513","3,433,734,906","-2,611,916,814","80,042","38,046,185,219","46,113,091,018","3,683,149,599","154,358,000,000","45,889,856,265","7,843,671,044",14
94,BEL,USA,2016,"-5,807,245,489","17,055,562,825","11,241,259,433","116,500","60,319,781,659","20,904,495,382","5,360,772,198","100,507,000,000","74,866,480,183","14,546,809,185",13
163,BMU,USA,2016,"-8,495,629,963","20,459,829,797","1,480,245,054","71,989","50,180,677,720","15,512,476,989","3,312,589,097","33,681,340,176","60,570,633,544","10,389,955,823",16
204,BRA,USA,2016,"-131,783,852","2,152,134,918","725,127,245","78,336","38,862,562,248","19,868,018,473","3,604,647,647","20,002,953,197","53,832,993,115","14,970,430,867",18
217,CAN,USA,2016,"-5,877,150,752","23,459,201,220","16,072,378,000","436,390","228,876,000,000","343,952,000,000","20,080,578,365","623,652,000,000","287,541,000,000","58,665,121,000",NaN
231,CHL,USA,2016,0,"202,989,684","208,708,220","5,985","3,692,185,585","1,313,330,648","275,401,044","2,527,248,868","4,044,031,401","351,845,816",1
315,CHN,USA,2016,"-3,165,530,873","2,359,320,751","-1,046,732,590","19,867","29,447,116,804","26,568,223,047","914,184,217","40,038,151,211","35,970,331,218","6,523,214,414",6
412,DNK,USA,2016,"-428,193,158","1,397,294,324","909,934,557","48,443","13,066,771,319","3,646,968,487","2,229,114,915","8,501,348,316","16,284,943,185","3,218,171,866",15
498,FRA,USA,2016,0,"15,554,756,670","16,683,047,003","409,385","241,543,000,000","58,385,420,416","18,837,937,565","182,879,000,000","292,313,000,000","50,769,400,314",NaN
530,IDN,USA,2016,"-4,650,218","5,266,078","-851,742",11,"31,921,039","9,283,335","506,167","414,712","32,302,873","381,834",0


In [9]:
# Calculate the total sum of profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
total_profit_loss_before_income_tax_corrected = misalignment_2016['profit_loss_before_income_tax_corrected'].sum()
total_n_employees = misalignment_2016['n_employees'].sum()
total_unrelated_party_revenues = misalignment_2016['unrelated_party_revenues'].sum()
total_tangible_assets_except_cash = misalignment_2016['tangible_assets_except_cash'].sum()
total_payroll = misalignment_2016['payroll'].sum()
total_stated_capital = misalignment_2016['stated_capital'].sum()
total_total_revenues = misalignment_2016['total_revenues'].sum()
total_related_party_revenues = misalignment_2016['related_party_revenues'].sum()
total_holding_or_managing_ip = misalignment_2016['holding_or_managing_ip'].sum()

misalignment_2016['total_profit_loss_before_income_tax_corrected'] = total_profit_loss_before_income_tax_corrected
misalignment_2016['total_n_employees'] = total_n_employees
misalignment_2016['total_unrelated_party_revenues'] = total_unrelated_party_revenues
misalignment_2016['total_tangible_assets_except_cash'] = total_tangible_assets_except_cash
misalignment_2016['total_payroll'] = total_payroll
misalignment_2016['total_stated_capital'] = total_stated_capital
misalignment_2016['total_total_revenues'] = total_total_revenues
misalignment_2016['total_related_party_revenues'] = total_related_party_revenues
misalignment_2016['total_holding_or_managing_ip'] = total_holding_or_managing_ip

# Group by iso_partner and calculate the total sum of profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
total_profit_loss_by_partner = misalignment_2016.groupby('iso_partner')['profit_loss_before_income_tax_corrected'].sum().reset_index()
total_profit_loss_by_partner = total_profit_loss_by_partner.rename(columns={'profit_loss_before_income_tax_corrected': 'total_profit_loss_by_partner'})

total_n_employees_by_partner = misalignment_2016.groupby('iso_partner')['n_employees'].sum().reset_index()
total_n_employees_by_partner = total_n_employees_by_partner.rename(columns={'n_employees': 'total_n_employees_by_partner'})

total_unrelated_party_revenues_by_partner = misalignment_2016.groupby('iso_partner')['unrelated_party_revenues'].sum().reset_index()
total_unrelated_party_revenues_by_partner = total_unrelated_party_revenues_by_partner.rename(columns={'unrelated_party_revenues': 'total_unrelated_party_revenues_by_partner'})

total_tangible_assets_except_cash_by_partner = misalignment_2016.groupby('iso_partner')['tangible_assets_except_cash'].sum().reset_index()
total_tangible_assets_except_cash_by_partner = total_tangible_assets_except_cash_by_partner.rename(columns={'tangible_assets_except_cash': 'total_tangible_assets_except_cash_by_partner'})

total_payroll_by_partner = misalignment_2016.groupby('iso_partner')['payroll'].sum().reset_index()
total_payroll_by_partner = total_payroll_by_partner.rename(columns={'payroll': 'total_payroll_by_partner'})

total_stated_capital_by_partner = misalignment_2016.groupby('iso_partner')['stated_capital'].sum().reset_index()
total_stated_capital_by_partner = total_stated_capital_by_partner.rename(columns={'stated_capital': 'total_stated_capital_by_partner'})

total_total_revenues_by_partner = misalignment_2016.groupby('iso_partner')['total_revenues'].sum().reset_index()
total_total_revenues_by_partner = total_total_revenues_by_partner.rename(columns={'total_revenues': 'total_total_revenues_by_partner'})

total_related_party_revenues_by_partner = misalignment_2016.groupby('iso_partner')['related_party_revenues'].sum().reset_index()
total_related_party_revenues_by_partner = total_related_party_revenues_by_partner.rename(columns={'related_party_revenues': 'total_related_party_revenues_by_partner'})

total_holding_or_managing_ip_by_partner = misalignment_2016.groupby('iso_partner')['holding_or_managing_ip'].sum().reset_index()
total_holding_or_managing_ip_by_partner = total_holding_or_managing_ip_by_partner.rename(columns={'holding_or_managing_ip': 'total_holding_or_managing_ip_by_partner'})

# Merge the total profit loss by partner back into the misalignment_2016 dataframe
misalignment_2016 = misalignment_2016.merge(total_profit_loss_by_partner, on='iso_partner', how='left')
misalignment_2016 = misalignment_2016.merge(total_n_employees_by_partner, on='iso_partner', how='left')
misalignment_2016 = misalignment_2016.merge(total_unrelated_party_revenues_by_partner, on='iso_partner', how='left')
misalignment_2016 = misalignment_2016.merge(total_tangible_assets_except_cash_by_partner, on='iso_partner', how='left')
misalignment_2016 = misalignment_2016.merge(total_payroll_by_partner, on='iso_partner', how='left')
misalignment_2016 = misalignment_2016.merge(total_stated_capital_by_partner, on='iso_partner', how='left')
misalignment_2016 = misalignment_2016.merge(total_total_revenues_by_partner, on='iso_partner', how='left')
misalignment_2016 = misalignment_2016.merge(total_related_party_revenues_by_partner, on='iso_partner', how='left')
misalignment_2016 = misalignment_2016.merge(total_holding_or_managing_ip_by_partner, on='iso_partner', how='left')

misalignment_2016[misalignment_2016['iso_partner'] == 'USA']

,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,total_profit_loss_before_income_tax_corrected,total_n_employees,total_unrelated_party_revenues,total_tangible_assets_except_cash,total_payroll,total_stated_capital,total_total_revenues,total_related_party_revenues,total_holding_or_managing_ip,total_profit_loss_by_partner,total_n_employees_by_partner,total_unrelated_party_revenues_by_partner,total_tangible_assets_except_cash_by_partner,total_payroll_by_partner,total_stated_capital_by_partner,total_total_revenues_by_partner,total_related_party_revenues_by_partner,total_holding_or_managing_ip_by_partner
53,AUS,USA,2016,"-5,424,987,513","3,433,734,906","-2,611,916,814","80,042","38,046,185,219","46,113,091,018","3,683,149,599","154,358,000,000","45,889,856,265","7,843,671,044",14,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462
94,BEL,USA,2016,"-5,807,245,489","17,055,562,825","11,241,259,433","116,500","60,319,781,659","20,904,495,382","5,360,772,198","100,507,000,000","74,866,480,183","14,546,809,185",13,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462
163,BMU,USA,2016,"-8,495,629,963","20,459,829,797","1,480,245,054","71,989","50,180,677,720","15,512,476,989","3,312,589,097","33,681,340,176","60,570,633,544","10,389,955,823",16,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462
204,BRA,USA,2016,"-131,783,852","2,152,134,918","725,127,245","78,336","38,862,562,248","19,868,018,473","3,604,647,647","20,002,953,197","53,832,993,115","14,970,430,867",18,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462
217,CAN,USA,2016,"-5,877,150,752","23,459,201,220","16,072,378,000","436,390","228,876,000,000","343,952,000,000","20,080,578,365","623,652,000,000","287,541,000,000","58,665,121,000",NaN,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462
231,CHL,USA,2016,0,"202,989,684","208,708,220","5,985","3,692,185,585","1,313,330,648","275,401,044","2,527,248,868","4,044,031,401","351,845,816",1,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462
315,CHN,USA,2016,"-3,165,530,873","2,359,320,751","-1,046,732,590","19,867","29,447,116,804","26,568,223,047","914,184,217","40,038,151,211","35,970,331,218","6,523,214,414",6,"2,234,653,761,818","78,780,594",

In [10]:
# Final Misalignment
final_misalignment_2016 = misalignment_2016

# Calculate the shares for all variables
final_misalignment_2016['share_reported_total_profit_loss_by_partner'] = misalignment_2016['total_profit_loss_by_partner'] / misalignment_2016['total_profit_loss_before_income_tax_corrected']
final_misalignment_2016['share_reported_total_n_employees_by_partner'] = misalignment_2016['total_n_employees_by_partner'] / misalignment_2016['total_n_employees']
final_misalignment_2016['share_reported_total_unrelated_party_revenues_by_partner'] = misalignment_2016['total_unrelated_party_revenues_by_partner'] / misalignment_2016['total_unrelated_party_revenues']
final_misalignment_2016['share_reported_total_tangible_assets_except_cash_by_partner'] = misalignment_2016['total_tangible_assets_except_cash_by_partner'] / misalignment_2016['total_tangible_assets_except_cash']
final_misalignment_2016['share_reported_total_payroll_by_partner'] = misalignment_2016['total_payroll_by_partner'] / misalignment_2016['total_payroll']
final_misalignment_2016['share_reported_total_stated_capital_by_partner'] = misalignment_2016['total_stated_capital_by_partner'] / misalignment_2016['total_stated_capital']
final_misalignment_2016['share_reported_total_total_revenues_by_partner'] = misalignment_2016['total_total_revenues_by_partner'] / misalignment_2016['total_total_revenues']
final_misalignment_2016['share_reported_total_related_party_revenues_by_partner'] = misalignment_2016['total_related_party_revenues_by_partner'] / misalignment_2016['total_related_party_revenues']
final_misalignment_2016['share_reported_total_holding_or_managing_ip_by_partner'] = misalignment_2016['total_holding_or_managing_ip_by_partner'] / misalignment_2016['total_holding_or_managing_ip']

# Give me column 'share reported' with 2 decimals
pd.options.display.float_format = '{:,.0f}'.format

final_misalignment_2016[final_misalignment_2016['iso_partner'] == 'USA']

,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,total_profit_loss_before_income_tax_corrected,total_n_employees,total_unrelated_party_revenues,total_tangible_assets_except_cash,total_payroll,total_stated_capital,total_total_revenues,total_related_party_revenues,total_holding_or_managing_ip,total_profit_loss_by_partner,total_n_employees_by_partner,total_unrelated_party_revenues_by_partner,total_tangible_assets_except_cash_by_partner,total_payroll_by_partner,total_stated_capital_by_partner,total_total_revenues_by_partner,total_related_party_revenues_by_partner,total_holding_or_managing_ip_by_partner,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
53,AUS,USA,2016,"-5,424,987,513","3,433,734,906","-2,611,916,814","80,042","38,046,185,219","46,113,091,018","3,683,149,599","154,358,000,000","45,889,856,265","7,843,671,044",14,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462,0,0,0,0,0,0,0,0,0
94,BEL,USA,2016,"-5,807,245,489","17,055,562,825","11,241,259,433","116,500","60,319,781,659","20,904,495,382","5,360,772,198","100,507,000,000","74,866,480,183","14,546,809,185",13,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462,0,0,0,0,0,0,0,0,0
163,BMU,USA,2016,"-8,495,629,963","20,459,829,797","1,480,245,054","71,989","50,180,677,720","15,512,476,989","3,312,589,097","33,681,340,176","60,570,633,544","10,389,955,823",16,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462,0,0,0,0,0,0,0,0,0
204,BRA,USA,2016,"-131,783,852","2,152,134,918","725,127,245","78,336","38,862,562,248","19,868,018,473","3,604,647,647","20,002,953,197","53,832,993,115","14,970,430,867",18,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462,0,0,0,0,0,0,0,0,0
217,CAN,USA,2016,"-5,877,150,752","23,459,201,220","16,072,378,000","436,390","228,876,000,000","343,952,000,000","20,080,578,365","623,652,000,000","287,541,000,000","58,665,121,000",NaN,"2,234,653,761,818","78,780,594","27,782,051,560,972","21,358,997,508,086","1,978,432,767,300","37,074,353,840,708","39,284,072,704,211","11,502,147,534,610","4,230","307,998,782,454","18,845,856","9,340,430,753,791","5,095,086,910,196","867,196,059,161","13,425,902,277,042","12,525,719,027,196","3,185,369,515,182",462,0,0,0,0,0,0,0,0,0
231,CHL,USA,2016,0,"202,989,684","208,708,220","5,985","3,692,185,585","1,313,330,648","275,401,044","2,527,248,868","4,044,031,401","351,84

In [11]:
# Keep iso_partner and share_reported_total_profit_loss_by_partner	share_reported_total_n_employees_by_partner	share_reported_total_unrelated_party_revenues_by_partner	share_reported_total_tangible_assets_except_cash_by_partner	share_reported_total_payroll_by_partner	share_reported_total_stated_capital_by_partner	share_reported_total_total_revenues_by_partner	share_reported_total_related_party_revenues_by_partner	share_reported_total_holding_or_managing_ip_by_partner
shares_reported_2016 = final_misalignment_2016[['iso_partner', 'share_reported_total_profit_loss_by_partner', 'share_reported_total_n_employees_by_partner', 'share_reported_total_unrelated_party_revenues_by_partner', 'share_reported_total_tangible_assets_except_cash_by_partner', 'share_reported_total_payroll_by_partner', 'share_reported_total_stated_capital_by_partner', 'share_reported_total_total_revenues_by_partner', 'share_reported_total_related_party_revenues_by_partner', 'share_reported_total_holding_or_managing_ip_by_partner']]

# Drop duplicates
shares_reported_2016 = shares_reported_2016.drop_duplicates()

shares_reported_2016[shares_reported_2016['iso_partner'] == 'USA']

,iso_partner,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
53,USA,0,0,0,0,0,0,0,0,0


In [12]:
excluded_2016 = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Keep if year == 2016 and iso_parent == 'AUT', 'FIN', 'IRL0', 'KOR', 'NLD', 'NOR' 'SWE'
excluded_2016 = excluded_2016[excluded_2016['year'] == 2016]
excluded_2016 = excluded_2016[excluded_2016['iso_parent'].isin(['AUT', 'FIN', 'IRL', 'KOR', 'NLD', 'NOR', 'SWE'])]

# Sum by iso_parent: 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip' and 'profit_loss_before_income_tax_corrected'
excluded_2016 = excluded_2016.groupby('iso_parent').agg({'n_employees': 'sum', 'unrelated_party_revenues': 'sum', 'tangible_assets_except_cash': 'sum', 'payroll': 'sum', 'stated_capital': 'sum', 'total_revenues': 'sum', 'related_party_revenues': 'sum', 'holding_or_managing_ip': 'sum', 'profit_loss_before_income_tax_corrected': 'sum'}).reset_index()

# Mege iso_combinations_2016 with excluded_2016. 
excluded_jurisdictions_2016 = pd.merge(iso_combinations_2016, excluded_2016, on='iso_parent', how='left')

#Drop year_y
excluded_jurisdictions_2016 = excluded_jurisdictions_2016.drop(columns=['year_y'])
# Rename year_x to year
excluded_jurisdictions_2016 = excluded_jurisdictions_2016.rename(columns={'year_x': 'year'})

# Keep if iso_parent == 'AUT', 'FIN', 'IRL0', 'KOR', 'NLD', 'NOR' 'SWE'
excluded_jurisdictions_2016 = excluded_jurisdictions_2016[excluded_jurisdictions_2016['iso_parent'].isin(['AUT', 'FIN', 'IRL', 'KOR', 'NLD', 'NOR', 'SWE'])]

excluded_jurisdictions_2016[excluded_jurisdictions_2016['iso_partner'] == 'USA']


,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected
356,AUT,2016,USA,"1,642,504","595,515,485,298","321,669,523,402","11,220,274,768","251,168,133,984","773,401,121,859","177,917,146,079",451,"40,892,944,645"
1820,FIN,2016,USA,"574,440","212,203,720,328","85,594,738,489","8,679,716,707","272,663,996,317","286,935,000,000","74,736,508,828",161,"25,179,380,440"
2369,IRL,2016,USA,"675,541","243,658,701,311","132,780,656,599","3,460,389,921","1,979,041,000,000","391,436,009,406","168,692,880,218",445,"40,888,035,864"
2918,KOR,2016,USA,"2,813,456","1,516,352,000,000","1,003,830,000,000","50,888,795,676","333,624,000,000","2,179,797,000,000","670,960,000,000",452,"104,132,507,018"
3467,NLD,2016,USA,"3,672,779","1,387,430,000,000","812,177,000,000","29,960,639,369","2,186,659,000,000","2,208,954,000,000","823,648,000,000",925,"80,833,124,738"
3650,NOR,2016,USA,"672,690","342,765,588,000","366,517,883,000","12,025,313,494","706,424,401,000","462,498,166,000","118,626,206,000",205,"37,061,095,205"
4382,SWE,2016,USA,"3,213,592","1,033,521,226,758","572,745,042,454","16,525,422,223","577,420,358,478","1,547,807,469,018","514,285,642,155","1,166","98,068,647,975"


In [13]:
# Merge with share_reported_2016
excluded_jurisdictions_share_reported_2016 = pd.merge(excluded_jurisdictions_2016, shares_reported_2016, on='iso_partner', how='left')

# Multiply 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip' and 'profit_loss_before_income_tax_corrected' by share_reported
excluded_jurisdictions_share_reported_2016['n_employees'] = excluded_jurisdictions_share_reported_2016['n_employees'] * excluded_jurisdictions_share_reported_2016['share_reported_total_n_employees_by_partner']
excluded_jurisdictions_share_reported_2016['unrelated_party_revenues'] = excluded_jurisdictions_share_reported_2016['unrelated_party_revenues'] * excluded_jurisdictions_share_reported_2016['share_reported_total_unrelated_party_revenues_by_partner']
excluded_jurisdictions_share_reported_2016['tangible_assets_except_cash'] = excluded_jurisdictions_share_reported_2016['tangible_assets_except_cash'] * excluded_jurisdictions_share_reported_2016['share_reported_total_tangible_assets_except_cash_by_partner']
excluded_jurisdictions_share_reported_2016['payroll'] = excluded_jurisdictions_share_reported_2016['payroll'] * excluded_jurisdictions_share_reported_2016['share_reported_total_payroll_by_partner']
excluded_jurisdictions_share_reported_2016['stated_capital'] = excluded_jurisdictions_share_reported_2016['stated_capital'] * excluded_jurisdictions_share_reported_2016['share_reported_total_stated_capital_by_partner']
excluded_jurisdictions_share_reported_2016['total_revenues'] = excluded_jurisdictions_share_reported_2016['total_revenues'] * excluded_jurisdictions_share_reported_2016['share_reported_total_total_revenues_by_partner']
excluded_jurisdictions_share_reported_2016['related_party_revenues'] = excluded_jurisdictions_share_reported_2016['related_party_revenues'] * excluded_jurisdictions_share_reported_2016['share_reported_total_related_party_revenues_by_partner']
excluded_jurisdictions_share_reported_2016['holding_or_managing_ip'] = excluded_jurisdictions_share_reported_2016['holding_or_managing_ip'] * excluded_jurisdictions_share_reported_2016['share_reported_total_holding_or_managing_ip_by_partner']
excluded_jurisdictions_share_reported_2016['profit_loss_before_income_tax_corrected'] = excluded_jurisdictions_share_reported_2016['profit_loss_before_income_tax_corrected'] * excluded_jurisdictions_share_reported_2016['share_reported_total_profit_loss_by_partner']

excluded_jurisdictions_share_reported_2016[excluded_jurisdictions_share_reported_2016['iso_partner'] == 'USA']

,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
173,AUT,2016,USA,"392,919","200,214,557,266","76,732,729,496","4,918,124,195","90,956,644,490","246,598,798,975","49,271,829,599",49,"5,636,209,679",0,0,0,0,0,0,0,0,0
356,FIN,2016,USA,"137,417","71,343,692,926","20,418,216,327","3,804,534,704","98,741,037,666","91,489,169,571","20,697,299,889",18,"3,470,434,056",0,0,0,0,0,0,0,0,0
539,IRL,2016,USA,"161,603","81,918,976,436","31,674,191,877","1,516,774,567","716,679,006,259","124,809,296,324","46,717,289,659",49,"5,635,533,109",0,0,0,0,0,0,0,0,0
722,KOR,2016,USA,"673,034","509,803,274,365","239,458,855,273","22,305,818,926","120,816,757,603","695,027,854,263","185,813,607,718",49,"14,352,418,224",0,0,0,0,0,0,0,0,0
905,NLD,2016,USA,"878,600","466,459,210,627","193,740,946,872","13,132,489,928","791,864,645,122","704,324,558,106","228,098,554,861",101,"11,141,101,332",0,0,0,0,0,0,0,0,0
1088,NOR,2016,USA,"160,921","115,239,086,375","87,431,091,619","5,270,992,598","255,820,641,263","147,467,451,288","32,851,978,220",22,"5,108,071,950",0,0,0,0,0,0,0,0,0
1271,SWE,2016,USA,"768,754","347,473,743,252","136,625,596,196","7,243,501,657","209,103,827,919","493,518,113,845","142,424,690,840",127,"13,516,646,153",0,0,0,0,0,0,0,0,0


In [14]:
excluded_jurisdictions_dataset_2016 = excluded_jurisdictions_share_reported_2016.drop(columns=[
    'share_reported_total_profit_loss_by_partner',
    'share_reported_total_n_employees_by_partner',
    'share_reported_total_unrelated_party_revenues_by_partner',
    'share_reported_total_tangible_assets_except_cash_by_partner',
    'share_reported_total_payroll_by_partner',
    'share_reported_total_stated_capital_by_partner',
    'share_reported_total_total_revenues_by_partner',
    'share_reported_total_related_party_revenues_by_partner',
    'share_reported_total_holding_or_managing_ip_by_partner'
])

excluded_jurisdictions_dataset_2016[excluded_jurisdictions_dataset_2016['iso_partner'] == 'USA']

,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected
173,AUT,2016,USA,"392,919","200,214,557,266","76,732,729,496","4,918,124,195","90,956,644,490","246,598,798,975","49,271,829,599",49,"5,636,209,679"
356,FIN,2016,USA,"137,417","71,343,692,926","20,418,216,327","3,804,534,704","98,741,037,666","91,489,169,571","20,697,299,889",18,"3,470,434,056"
539,IRL,2016,USA,"161,603","81,918,976,436","31,674,191,877","1,516,774,567","716,679,006,259","124,809,296,324","46,717,289,659",49,"5,635,533,109"
722,KOR,2016,USA,"673,034","509,803,274,365","239,458,855,273","22,305,818,926","120,816,757,603","695,027,854,263","185,813,607,718",49,"14,352,418,224"
905,NLD,2016,USA,"878,600","466,459,210,627","193,740,946,872","13,132,489,928","791,864,645,122","704,324,558,106","228,098,554,861",101,"11,141,101,332"
1088,NOR,2016,USA,"160,921","115,239,086,375","87,431,091,619","5,270,992,598","255,820,641,263","147,467,451,288","32,851,978,220",22,"5,108,071,950"
1271,SWE,2016,USA,"768,754","347,473,743,252","136,625,596,196","7,243,501,657","209,103,827,919","493,518,113,845","142,424,690,840",127,"13,516,646,153"


In [15]:
final_misalignment_2016 = cbcr_sample[cbcr_sample['year'] == 2016].copy()

final_misalignment_2016 #[final_misalignment_2016['iso_parent'] == 'AUT']

# Concatenate excluded_jurisdictions_dataset_2016
final_misalignment_2016 = pd.concat([final_misalignment_2016, excluded_jurisdictions_dataset_2016])

# Ensure all required columns are included
#required_columns = [
#    'iso_parent', 'year', 'iso_partner', 'n_employees', 'unrelated_party_revenues',
#    'tangible_assets_except_cash', 'payroll', 'stated_capital', 'total_revenues',
#    'related_party_revenues', 'holding_or_managing_ip', 'profit_loss_before_income_tax_corrected'
#]

# Filter the dataframe to include only the required columns
#final_misalignment_2016 = final_misalignment_2016[required_columns]

final_misalignment_2016[final_misalignment_2016['iso_partner'] == 'USA']

,iso_parent,parent_jurisdiction,iso_partner,partner_jurisdiction,year,unrelated_party_revenues,profit_loss_before_income_tax,adjusted_profit_loss_before_income_tax,income_tax_paid_on_cash_basis,income_tax_accrued_current_year,n_employees,tangible_assets_except_cash,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,n_cbcr,n_cbcr_groups,n_entities,profit_loss_before_income_tax_corrected,ln_profit_loss_before_income_tax_corrected,ln_unrelated_party_revenues,ln_n_employees,ln_tangible_assets_except_cash,ln_stated_capital,ln_total_revenues,ln_related_party_revenues,ln_holding_or_managing_ip,etr_domestic,etr_domestic_corrected,etr_foreign,etr_foreign_corrected,etr_average,etr_average_corrected,cit,gdp_current_usd,population,gdp,wage_monthly,payroll,ln_wage_monthly,ln_gdp_current_usd,ln_population,gvt_health_expenditure,ln_gvt_health_expenditure,tax_revenue_pct_gdp,tax_revenue_current_usd,cthi_2021_share,cthi_2021_score,region_tjn,ukt,gbr_oct,nld_oct,oecd_oct,oecd,eu
715,AUS,Australia,USA,United States,2016,"38,046,185,219","-2,611,916,814",NaN,"273,368,137","369,823,101","80,042","46,113,091,018","154,358,000,000","45,889,856,265","7,843,671,044",14,69,69,"1,089","-2,611,916,814",0,24,11,25,26,25,23,3,0,1,0,0,0,0,0,"18,804,913,000,000","323,071,755",NaN,"3,835","3,683,149,599",8,31,20,"1,608,175,818,558",28,11,"2,041,182,100,441",0,47,Northern America,0,0,0,0,1,0
992,BEL,Belgium,USA,United States,2016,"60,319,781,659","11,241,259,433",NaN,"4,432,519,679","4,802,237,523","116,500","20,904,495,382","100,507,000,000","74,866,480,183","14,546,809,185",13,37,37,358,"11,241,259,433",23,25,12,24,25,25,23,3,0,1,0,0,0,0,0,"18,804,913,000,000","323,071,755",NaN,"3,835","5,360,772,198",8,31,20,"1,608,175,818,558",28,11,"2,041,182,100,441",0,47,Northern America,0,0,0,0,1,0
1600,BMU,Bermuda,USA,United States,2016,"50,180,677,720","1,480,245,054",NaN,"338,517,495","267,491,121","71,989","15,512,476,989","33,681,340,176","60,570,633,544","10,389,955,823",16,35,35,627,"1,480,245,054",21,25,11,23,24,25,23,3,0,1,0,0,0,0,0,"18,804,913,000,000","323,071,755",NaN,"3,835","3,312,589,097",8,31,20,"1,608,175,818,558",28,11,"2,041,182,100,441",0,47,Northern America,0,0,0,0,1,0
1862,BRA,Brazil,USA,United States,2016,"38,862,562,248","725,127,245",NaN,"609,464,549","1,302,156","78,336","19,868,018,473","20,002,953,197","53,832,993,115","14,970,430,867",18,41,41,242,"725,127,245",20,24,11,24,24,25,23,3,0,1,0,0,0,0,0,"18,804,913,000,000","323,071,755",NaN,"3,835","3,604,647,647",8,31,20,"1,608,175,818,558",28,11,"2,041,182,100,441",0,47,Northern America,0,0,0,0,1,0
1970,CAN,Canada,USA,United States,2016,"228,876,000,000","16,072,378,000",NaN,"2,117,891,000","1,781,351,000","436,390","343,952,000,000","623,652,000,000","287,541,000,000","58,665,121,000",NaN,150,150,"7,180","16,072,378,000",24,26,13,27,27,26,25,NaN,0,1,0,0,0,0,0,"18,804,913,000,000","323,071,755",NaN,"3,835","20,080,578,365",8,31,20,"1,608,175,818,558",28,11,"2,041,182,100,441",0,47,Northern America,0,0,0,0,1,0
2802,CHL,Chile,USA,United States,2016,"3,692,185,585","208,708,220",NaN,"42,768,069","58,215,874","5,985","1,313,330,648","2,527,248,868","4,044,031,401","351,845,816",1,17,17,45,"208,708,220",19,22,9,21,22,22,20,1,0,1,0,0,0,0,0,"18,804,913,000,000","323,071,755",NaN,"3,835","275,401,044",8,31,20,"1,608,175,818,558",28,11,"2,041,182,100,441",0,47,Northern America,0,0,0,0,1,0
3519,CHN,China (People’s Republic of),USA,United States,2016,"29,447,116,804","-1,046,732,590",NaN,"409,206,765","444,595,002","19,867","26,568,223,047","40,038,151,211","35,970,331,218","6,523,214,414",6,47,47,209,"-1,046,732,590",0,24,10,24,24,24,23,2,0,1,0,0,0,0,0,"18,804,913,000,000","323,071,755",NaN,"3,835","914,184,217",8,31,20,"1,608,175,818,558",28,11,"2,041,182,100,441",0,47,Northern America,0,0,0,0,1,0
5576,DNK,Denmark,USA,United States,2016,"13,066,771,319","909,934,557",NaN,"205,290,752","178,426,590","48,443","3,646,968,487","8,501,348,316","16,284,943,185","3,218,171,866",15,29,

In [16]:
# Initialize a list to store the aggregate results
results_sample = []

# Start the estimates
misalignment_final_estimates_2016 = final_misalignment_2016[final_misalignment_2016['year'] == 2016].copy()
misalignment_final_estimates_2016 = calculate_misalignment(misalignment_final_estimates_2016, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Perform the groupby operation on 'iso_partner'
country_results_2016 = misalignment_final_estimates_2016.groupby(['iso_partner']).agg(
    negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
    positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
    theoretical_profit=('theoretical_profit', 'sum'),
    reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
).reset_index()

# Convert results to millions
country_results_2016['negative_misalignment'] = -country_results_2016['negative_misalignment'] / 1e6
country_results_2016['positive_misalignment'] = country_results_2016['positive_misalignment'] / 1e6
country_results_2016['theoretical_profit'] = country_results_2016['theoretical_profit'] / 1e6
country_results_2016['reported_profit'] = country_results_2016['reported_profit'] / 1e6

# Merge the unique columns back into the result
country_results_2016 = country_results_2016.merge(unique_columns, on='iso_partner', how='left')

# Calculate other relevant variables
country_results_2016['tax_revenue_loss'] = country_results_2016['negative_misalignment'] * country_results_2016['cit']
country_results_2016['tax_revenue_gain'] = country_results_2016['positive_misalignment'] * country_results_2016['etr_average_corrected']

country_results_2016['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
    country_results_2016['gvt_health_expenditure'] == 0, 
    np.nan, 
    country_results_2016['tax_revenue_loss'] / (country_results_2016['gvt_health_expenditure'] / 1e6)
)
    
country_results_2016['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
    country_results_2016['tax_revenue_current_usd'] == 0, 
    np.nan, 
    country_results_2016['tax_revenue_loss'] / (country_results_2016['tax_revenue_current_usd'] / 1e6)
)

# Calculate totals
total_positive_misalignment = country_results_2016['positive_misalignment'].sum()
total_negative_misalignment = country_results_2016['negative_misalignment'].sum()
total_profits = country_results_2016['reported_profit'].sum()
misaligned_of_total_profits = total_positive_misalignment / total_profits
total_tax_revenue_loss = country_results_2016['tax_revenue_loss'].sum()
total_tax_revenue_gain = country_results_2016['tax_revenue_gain'].sum()
average_tax_revenue_loss_pct_of_gvt_health_expenditure = country_results_2016['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
average_tax_revenue_loss_pct_of_total_tax_revenues = country_results_2016['tax_revenue_loss_pct_of_total_tax_revenues'].mean()


print(f"Year {2016}: Positive Misalignment: {total_positive_misalignment}, Negative Misalignment: {total_negative_misalignment}, Shifted of total profits: {misaligned_of_total_profits}, "
        f"Total tax revenue loss: {total_tax_revenue_loss}, Total tax revenue gain: {total_tax_revenue_gain}")

# Calculate countries' fractions of totals
country_results_2016['tax_revenue_loss_caused_pct_of_total'] = country_results_2016['positive_misalignment'] / total_positive_misalignment
country_results_2016['tax_revenue_loss_caused_usd'] = country_results_2016['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss
country_results_2016['tax_revenue_loss_suffered_pct_of_total'] = country_results_2016['tax_revenue_loss'] / total_tax_revenue_loss

#country_results_2016 = country_results_2016[['iso_partner', 'partner_jurisdiction', 'negative_misalignment',
#   'tax_revenue_loss', 'tax_revenue_loss_suffered_pct_of_total', 'tax_revenue_loss_pct_of_gvt_health_expenditure',
#   'tax_revenue_loss_pct_of_total_tax_revenues', 'positive_misalignment', 'tax_revenue_gain', 
#   'tax_revenue_loss_caused_usd', 'tax_revenue_loss_caused_pct_of_total', 'etr_average_corrected', 'cit',
#   'region_tjn', 'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

country_results_2016 = country_results_2016.sort_values(by='iso_partner')
country_results_2016.to_csv(f'{output_tables}/Scaling_Mario/SOTJ_sample_countries_2016.csv', index=False) # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE
    
# Append aggregate results to the list
results_sample.append({
    'year': 2016,
    'total_positive_misalignment': total_positive_misalignment,
    'total_negative_misalignment': total_negative_misalignment,
    'total_profits': total_profits,
    'misaligned_of_total_profits': misaligned_of_total_profits,
    'total_tax_revenue_loss': total_tax_revenue_loss,
    'total_tax_revenue_gain': total_tax_revenue_gain,
    'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_tax_revenue_loss_pct_of_gvt_health_expenditure,
    'average_tax_revenue_loss_pct_of_total_tax_revenues': average_tax_revenue_loss_pct_of_total_tax_revenues
})

# Convert aggregate results to a DataFrame
results_sample_df = pd.DataFrame(results_sample)

# Save the aggregated results to a CSV or Excel file
results_sample_df.to_csv(f'{output_tables}/Scaling_Mario/SOTJ_sample_aggregate_results_2016.csv', index=False)  # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE

Year 2016: Positive Misalignment: 530615.3114510451, Negative Misalignment: 530615.3114510451, Shifted of total profits: 0.19935132361706898, Total tax revenue loss: 147413.25389962248, Total tax revenue gain: 40946.056384616044


/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_61006/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


In [17]:
# Open the CSV file f'{output_tables}/Scaling_Mario/SOTJ_sample_2016.csv'
sotj_2016_scaled_up = pd.read_csv(f'{output_tables}/Scaling_Mario/SOTJ_sample_countries_2016.csv')

#Show me 2 decimals
pd.options.display.float_format = '{:,.2f}'.format
sotj_2016_scaled_up[sotj_2016_scaled_up['iso_partner'] == 'FRA']

,iso_partner,negative_misalignment,positive_misalignment,theoretical_profit,reported_profit,partner_jurisdiction,etr_average_corrected,cit,tax_revenue_current_usd,gvt_health_expenditure,region_tjn,ukt,oecd,oecd_oct,nld_oct,tax_revenue_loss,tax_revenue_gain,tax_revenue_loss_pct_of_gvt_health_expenditure,tax_revenue_loss_pct_of_total_tax_revenues,tax_revenue_loss_caused_pct_of_total,tax_revenue_loss_caused_usd,tax_revenue_loss_suffered_pct_of_total
54,FRA,"19,589.60",0.00,"134,300.32","112,603.82",France,0.19,0.34,"570,384,960,215.54","212,992,618,347.21",Europe,0.00,1.00,0.00,0.00,"6,745.35",0.00,0.03,0.01,0.00,0.00,0.05


In [18]:
sotj_aggregate_scaled_up = pd.read_csv(f'{output_tables}/Scaling_Mario/SOTJ_sample_aggregate_results_2016.csv')
sotj_aggregate_scaled_up


,year,total_positive_misalignment,total_negative_misalignment,total_profits,misaligned_of_total_profits,total_tax_revenue_loss,total_tax_revenue_gain,average_tax_revenue_loss_pct_of_gvt_health_expenditure,average_tax_revenue_loss_pct_of_total_tax_revenues
0,2016,"530,615.31","530,615.31","2,661,709.50",0.20,"147,413.25","40,946.06",0.09,0.01
